# FightFlow — Colab Trainer

Run this notebook top-to-bottom at the start of every Colab session.
Training outputs are saved to Google Drive and persist after the session ends.

> **First time?** Follow the setup guide in the repo README before running this.

## 1. Check GPU
Make sure Colab assigned you a GPU. If `CUDA available: False`, go to
**Runtime → Change runtime type → T4 GPU** and re-run.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

## 2. Mount Google Drive
Videos and training outputs live here so they persist between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone / update repo
Pulls the latest code + config files from GitHub. Already cloned? It just runs `git pull`.
The repo includes `annotations.csv`, `clip_manifest.csv`, and directory structure — no need to upload those to Drive.

In [ ]:
import os

REPO_URL = "https://github.com/krishmula/fightflow.git"
REPO_DIR = "/content/fightflow"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print(f"Repo ready at {REPO_DIR}")

## 4. Symlink large binary dirs from Drive

The repo already contains `annotations.csv`, `clip_manifest.csv`, and the directory skeleton.
We only need to symlink the two directories that hold large binaries:

- `data/downloaded-videos/` → your videos on Drive  
- `data/processed/clips/` → generated clips on Drive (persisted between sessions)  
- `runs/` → training outputs (checkpoints, metrics) on Drive

In [ ]:
DRIVE_ROOT   = "/content/drive/MyDrive/fightflow"
DRIVE_DATA   = f"{DRIVE_ROOT}/data"          # matches your Drive folder structure
DRIVE_VIDEOS = f"{DRIVE_DATA}/downloaded-videos"
DRIVE_CLIPS  = f"{DRIVE_DATA}/processed/clips"
DRIVE_RUNS   = f"{DRIVE_ROOT}/runs"

# Create Drive dirs if they don't exist yet (first run only)
for d in [DRIVE_CLIPS, DRIVE_RUNS]:
    os.makedirs(d, exist_ok=True)

# Map repo path → Drive target
LINKS = {
    f"{REPO_DIR}/data/downloaded-videos": DRIVE_VIDEOS,
    f"{REPO_DIR}/data/processed/clips":   DRIVE_CLIPS,
    f"{REPO_DIR}/runs":                   DRIVE_RUNS,
}

for link, target in LINKS.items():
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link) and not os.listdir(link):
        os.rmdir(link)  # remove empty placeholder from git clone
    !ln -s {target} {link}
    print(f"{link}  →  {os.readlink(link)}")

# Sanity checks
videos = os.listdir(f"{REPO_DIR}/data/downloaded-videos")
print(f"\nVideos found: {len(videos)} — {videos[:3]} ...")
print(f"annotations.csv present: {os.path.exists(f'{REPO_DIR}/data/annotations.csv')}")
print(f"clip_manifest.csv present: {os.path.exists(f'{REPO_DIR}/data/processed/clip_manifest.csv')}")

## 5. Install dependencies
Takes ~1 minute on first run. Cached for the session.

In [ ]:
%pip install -q -r {REPO_DIR}/requirements.txt
print("Dependencies installed.")

## 6. Prepare clip data
Extracts frame clips from your videos and saves them to `data/processed/clips/` on Drive.

**Only run this once.** The cell auto-detects whether clips already exist and skips if so.
Delete `MyDrive/fightflow/processed/clips/` from Drive if you ever need to regenerate.

In [ ]:
clips_exist = os.path.isdir(DRIVE_CLIPS) and len(os.listdir(DRIVE_CLIPS)) > 0

if clips_exist:
    n = len(os.listdir(DRIVE_CLIPS))
    print(f"Clips already on Drive ({n} clips). Skipping prepare_data.")
else:
    print("No clips found — running prepare_data. This may take a few minutes...")
    %cd {REPO_DIR}
    !python main.py --model cnn_lstm --task prepare_data
    print(f"Done. {len(os.listdir(DRIVE_CLIPS))} clips saved to Drive.")

## 7. Train
Runs the full training loop. Checkpoints and metrics are saved to
`runs/cnn_lstm/<run_name>/` on Drive in real time.

Add CLI flags below to override any value in `hparams.yaml` — e.g. `--epochs 30 --lr 0.0005`.

In [ ]:
%cd {REPO_DIR}

# Uncomment and modify flags to override hparams.yaml:
# --epochs 30
# --lr 0.0005
# --freeze-backbone false
# --cnn-backbone vgg_16

!python main.py --model cnn_lstm --task train

## 8. (Optional) Evaluate a checkpoint
Run test or validation against a specific saved checkpoint.

In [ ]:
# List available runs to find your run folder name
!ls {DRIVE_RUNS}/cnn_lstm/

In [ ]:
%cd {REPO_DIR}

RUN_NAME = "REPLACE_WITH_RUN_FOLDER_NAME"  # e.g. cnn_lstm_20240508_143012

!python main.py --model cnn_lstm --task test \
    --checkpoint runs/cnn_lstm/{RUN_NAME}/checkpoints/best.pt \
    --report-dir runs/cnn_lstm/{RUN_NAME}/reports